# Feature maps of the sites around each sensor

One map per built environment feature. Every map shows the eight site networks together, so the
colour scale compares places directly.

Each map has three layers:

1. The street networks of the eight sites, coloured by the feature.
2. A dot at each sensor location.
3. A CARTO Positron basemap, drawn with `CARTO_API_KEY`.

The feature values come from `built_environment_features.csv`, written by
`built_environment_features.ipynb`. The values are per sampling point. `tag_streets` averages the
points of a street into one street value.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PARENT_PATH_SITES = Path("/Users/legariapena.j/Documents/open-sensing-analysis/open-sensing-analysis/data/2026_09_16_site_built_environment_comparison")
PATH_PROJECT_ENV = Path("/Users/legariapena.j/Documents/open-sensing-analysis/open-sensing-analysis/.env")
PATH_SV_PAPER_ENV = Path("/Users/legariapena.j/Documents/SV-paper-refinement/.env")

FEATURES_FILE_NAME = "built_environment_features.csv"
NON_FEATURE_COLUMNS = ["site_name", "point_index", "latitude", "longitude", "n_views"]

load_dotenv(PATH_PROJECT_ENV)
load_dotenv(PATH_SV_PAPER_ENV)
CARTO_API_KEY = os.environ.get("CARTO_API_KEY")

if CARTO_API_KEY is None:
    raise SystemExit("CARTO_API_KEY not found in the environment")

all_features_data = pd.read_csv(PARENT_PATH_SITES/FEATURES_FILE_NAME)
feature_names = [column for column in all_features_data.columns if column not in NON_FEATURE_COLUMNS]

print(f"rows: {len(all_features_data)}")
print(f"features: {len(feature_names)}")
print(feature_names)

In [ ]:
PATH_SENSOR_LOCATIONS = "/Users/legariapena.j/Documents/open-sensing-analysis/open-sensing-analysis/data/spatial/data-1775857265051.csv"
EPA_SENSOR_DATA_PATH = "/Users/legariapena.j/Documents/open-sensing-analysis/open-sensing-analysis/data/2026_08_26_sensor_comparisons/data/boston_pm25_combined.csv"

ROXBURY_KEY = "DUDLEY SQUARE ROXBURY"
VON_HILLERN_KEY = "VON HILLERN ST"
CARTER_FIELD_KEY = "Carter Field"
ROBINSON_HALL_KEY = "Robinson Hall"
GAINSBOROUGH_KEY = "Gainsborough Parking Garage"
WEST_VILLAGE_B_KEY = "West Village B"
WORLD_SERIES_KEY = "World Series Way "
AMY_CARTER_FIELD_KEY = "Carter Field — 2"

LATITUDE_AMY = 42.3382
LONGITUDE_AMY = -71.0851

sensor_locations = pd.read_csv(PATH_SENSOR_LOCATIONS)
epa_data = pd.read_csv(EPA_SENSOR_DATA_PATH)

site_name_to_location = {}

for key in [ROXBURY_KEY, VON_HILLERN_KEY]:
    sub_data = epa_data[epa_data["site_name"] == key]
    site_name_to_location[key] = [list(sub_data["lon"])[0], list(sub_data["lat"])[0]]

for key in [CARTER_FIELD_KEY, ROBINSON_HALL_KEY, GAINSBOROUGH_KEY, WEST_VILLAGE_B_KEY, WORLD_SERIES_KEY]:
    sub_data = sensor_locations[sensor_locations["ecohub_location"] == key]
    site_name_to_location[key] = [list(sub_data["lon"])[0], list(sub_data["lat"])[0]]

site_name_to_location[AMY_CARTER_FIELD_KEY] = [LONGITUDE_AMY, LATITUDE_AMY]

for site_name, location in site_name_to_location.items():
    print(f"{site_name}: {location}")

In [ ]:
from alivestreets.sampling.street_sampler import StreetSampler
from tqdm import tqdm

N_SAMPLING_POINTS = 300
COORDINATE_TOLERANCE = 1e-6

site_names = sorted(site_name_to_location.keys())
site_name_to_sampler = {}

for site_name in tqdm(site_names):
    sampler = StreetSampler(N_SAMPLING_POINTS)
    sampler.open_streets(str(PARENT_PATH_SITES/site_name/"network.geojson"))
    sampler.sample_streets_no_intersections()
    site_name_to_sampler[site_name] = sampler

for site_name in site_names:
    sampler = site_name_to_sampler[site_name]
    sampling_points = sampler.get_all_sampling_points()
    site_features = all_features_data[all_features_data["site_name"] == site_name].sort_values("point_index")

    if len(sampling_points) != len(site_features):
        raise SystemExit(f"{site_name}: {len(sampling_points)} sampling points against {len(site_features)} feature rows")

    largest_difference = 0.0
    for (longitude, latitude), (_, row) in zip(sampling_points, site_features.iterrows()):
        largest_difference = max(largest_difference, abs(longitude - row["longitude"]), abs(latitude - row["latitude"]))

    if largest_difference > COORDINATE_TOLERANCE:
        raise SystemExit(f"{site_name}: sampling points do not match the feature rows ({largest_difference})")

    print(f"{site_name}: {len(sampler.streets)} streets, {len(sampling_points)} points, max coordinate difference {largest_difference:.2e}")

In [ ]:
for site_name in tqdm(site_names):
    sampler = site_name_to_sampler[site_name]
    site_features = all_features_data[all_features_data["site_name"] == site_name].sort_values("point_index")

    for feature_name in feature_names:
        values = [None if pd.isna(value) else float(value) for value in site_features[feature_name]]
        sampler.tag_streets(feature_name, values, method = "mean")

print("tagged")

In [ ]:
import cmcrameri.cm as cmc
from alivestreets.visualization.map_visualizer import MapVisualizer

OUTPUT_PATH_MAPS = PARENT_PATH_SITES/"feature_maps"
OUTPUT_PATH_MAPS.mkdir(exist_ok = True)

FIGURE_SIZE = (10, 10)
EDGE_WIDTH = 1.4
MIN_PERCENTILE = 1
MAX_PERCENTILE = 99
BASEMAP_ALPHA = 0.8
SENSOR_COLOR = "#282A36"
SENSOR_SIZE = 55

sensor_points = [site_name_to_location[site_name] for site_name in site_names]
sensor_weights = [1.0 for site_name in site_names]

for feature_name in tqdm(feature_names):
    visualizer = MapVisualizer()
    visualizer.initialize_map(figsize = FIGURE_SIZE)

    for site_name in site_names:
        visualizer.add_street_sampler(
            site_name_to_sampler[site_name],
            feature_name,
            "continuous",
            cmap = cmc.hawaii,
            edge_width = EDGE_WIDTH
        )

    visualizer.draw_networks(
        colorbar_label = feature_name,
        colorbar_orientation = "horizontal",
        min_percentile = MIN_PERCENTILE,
        max_percentile = MAX_PERCENTILE
    )

    visualizer.add_points(
        sensor_points,
        sensor_weights,
        color = SENSOR_COLOR,
        min_size = SENSOR_SIZE,
        max_size = SENSOR_SIZE,
        alpha = 1.0,
        label_family = "Sensors"
    )

    visualizer.add_basemap(alpha = BASEMAP_ALPHA, api_key = CARTO_API_KEY)

    file_name = feature_name.replace(" ", "_") + ".png"
    visualizer.finalize_map(
        title = feature_name,
        save_path = str(OUTPUT_PATH_MAPS/file_name),
        show_legend = True
    )

print(f"maps written to {OUTPUT_PATH_MAPS}")